In [ ]:
# Full Coverage Test Suite — CRUD, Scoring, Aggregations, Edge Cases
# Test_Script.ipynb

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from CRUD_Python_Module import (
    AnimalShelter,
    AnimalShelterValidationError,
    AnimalShelterConnectionError,
    AnimalShelterCrudError,
    AnimalShelterAnalyticsError,
)
from rescue_scoring import (
    compute_rescue_score,
    rank_animals,
    normalize_breeds,
)

import traceback

# ---------------------------------------------------------
# Helper
# ---------------------------------------------------------
def section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80 + "\n")


# ---------------------------------------------------------
# 0. AUTHENTICATION TESTS
# ---------------------------------------------------------
section("0. AUTHENTICATION TESTS")

print("---- FAILED LOGIN ATTEMPT ----")
try:
    AnimalShelter("incorrectuser", "incorrectpassword")
    print("Unexpected success: bad credentials accepted!")
except AnimalShelterConnectionError as e:
    print("Failed Login Attempt (expected):", e)

print("\n---- EMPTY PASSWORD LOGIN ----")
try:
    AnimalShelter("aacuser", "")
    print("Unexpected success: empty password accepted!")
except AnimalShelterValidationError as e:
    print("Empty Password Login Failed (expected):", e)

print("\n---- VALID LOGIN ----")
try:
    shelter = AnimalShelter("aacuser", "CS340AndreMiranda")
    print("Successful login and initialization.")
except Exception as e:
    print("Unexpected failure on valid login:", e)
    traceback.print_exc()
    raise SystemExit("Cannot continue tests without valid shelter instance.")


# ---------------------------------------------------------
# 1. CRUD TESTS — CREATE / READ / UPDATE / DELETE
# ---------------------------------------------------------
section("1. CRUD TESTS — STANDARD, REALISTIC, EDGE, NEGATIVE CASES")

# 1.1 Standard Test Record
standard_record = {
    "name": "NotebookTestAnimal",
    "animal_type": "Dog",
    "breed": "TestBreed",
    "age_upon_outcome": "3 years",
    "age_upon_outcome_in_weeks": 156,
    "outcome_type": "Adoption",
}

print("---- INSERTING STANDARD TEST RECORD ----")
print("Create Result:", shelter.create(standard_record))

print("\n---- READING STANDARD TEST RECORD ----")
for doc in shelter.read({"name": "NotebookTestAnimal"}):
    print(doc)

# 1.2 Realistic Record
realistic_record = {
    "name": "Yeti Miranda",
    "animal_type": "Cat",
    "breed": "Domestic Mix",
    "age_upon_outcome": "1 year",
    "age_upon_outcome_in_weeks": 52,
    "outcome_type": "Transfer",
}

print("\n---- INSERTING REALISTIC RECORD ----")
print("Create Result:", shelter.create(realistic_record))

print("\n---- READING REALISTIC RECORD ----")
for doc in shelter.read({"name": "Yeti Miranda"}):
    print(doc)

# 1.3 Edge Case Record
edge_case_record = {
    "name": "EdgeCaseRecord",
    "animal_type": "Unknown",
    "notes": None,
    "list_field": [],
    "dict_field": {},
    "numeric_field": 0,
    "breed": "Unknown Mix",
    "age_upon_outcome_in_weeks": None,
    "outcome_type": "Other",
}

print("\n---- INSERTING EDGE CASE RECORD ----")
print("Create Result:", shelter.create(edge_case_record))

print("\n---- READING EDGE CASE RECORD ----")
for doc in shelter.read({"numeric_field": 0}):
    print(doc)

# 1.4 Negative CREATE tests (aligned with module behavior)
section("1.4 NEGATIVE CREATE TESTS (Aligned with module behavior)")

invalid_records = [
    {"name": "MissingType"},  # allowed by schema (warn only)
    {
        "name": "BadTypeRecord",
        "animal_type": 123,  # allowed (schema warns only)
        "breed": "Test",
        "age_upon_outcome_in_weeks": "not_a_number",
        "outcome_type": "Adoption",
    },
    {
        "name": "ExtraFieldRecord",
        "animal_type": "Dog",
        "breed": "Test",
        "age_upon_outcome_in_weeks": 20,
        "outcome_type": "Adoption",
        "extra_field_not_in_schema": "extra",  # NOT allowed
    },
]

for idx, rec in enumerate(invalid_records, start=1):
    print(f"\n-- Attempting create #{idx} --")
    try:
        result = shelter.create(rec)
        print("Create result:", result)
    except AnimalShelterValidationError as e:
        print("Validation failure (expected for extra fields):", e)
    except Exception as e:
        print("Unexpected exception:", e)

# 1.5 READ edge cases
section("1.5 READ EDGE CASES")

print("\n---- EMPTY QUERY READ ----")
try:
    results = shelter.read({})
    print("Empty query returned:", len(results), "documents.")
except Exception as e:
    print("Empty query read failed:", e)

print("\n---- READ NO MATCH ----")
results = shelter.read({"name": "NoSuchName"})
print("No-match returned:", len(results), "documents (expected 0).")

# 1.6 UPDATE edge cases
section("1.6 UPDATE EDGE CASES")

print("\n---- UPDATE STANDARD RECORD ----")
print("Modified:", shelter.update({"name": "NotebookTestAnimal"}, {"outcome_type": "UpdatedOutcome"}))

print("\n---- READ UPDATED STANDARD RECORD ----")
for doc in shelter.read({"name": "NotebookTestAnimal"}):
    print(doc)

print("\n---- UPDATE REALISTIC RECORD ----")
print("Modified:", shelter.update({"name": "Yeti Miranda"}, {"outcome_type": "UpdatedTransfer"}))

print("\n---- READ UPDATED REALISTIC RECORD ----")
for doc in shelter.read({"name": "Yeti Miranda"}):
    print(doc)

print("\n---- UPDATE EDGE CASE RECORD ----")
print("Modified:", shelter.update({"numeric_field": 0}, {"notes": "Updated edge case note"}))

print("\n---- READ UPDATED EDGE CASE RECORD ----")
for doc in shelter.read({"numeric_field": 0}):
    print(doc)

print("\n---- UPDATE WITH EMPTY UPDATE DICT (EXPECTED FAILURE) ----")
try:
    shelter.update({"name": "NotebookTestAnimal"}, {})
    print("Unexpected success on empty update!")
except AnimalShelterValidationError as e:
    print("Empty update failed (expected):", e)

print("\n---- UPDATE WITH NO MATCH ----")
print("Modified:", shelter.update({"name": "NoSuchName"}, {"outcome_type": "X"}))

# 1.7 DELETE edge cases
section("1.7 DELETE EDGE CASES")

print("\n---- DELETE STANDARD RECORD ----")
print("Deleted:", shelter.delete({"name": "NotebookTestAnimal"}))

print("\n---- VERIFY DELETE ----")
print("Remaining:", shelter.read({"name": "NotebookTestAnimal"}))

print("\n---- DELETE REALISTIC RECORD ----")
print("Deleted:", shelter.delete({"name": "Yeti Miranda"}))

print("\n---- VERIFY DELETE ----")
print("Remaining:", shelter.read({"name": "Yeti Miranda"}))

print("\n---- DELETE EDGE CASE RECORD ----")
print("Deleted:", shelter.delete({"numeric_field": 0}))

print("\n---- VERIFY DELETE ----")
print("Remaining:", shelter.read({"numeric_field": 0}))

print("\n---- DELETE NO MATCH ----")
print("Deleted:", shelter.delete({"name": "NoSuchName"}))


# ---------------------------------------------------------
# 2. SCORING ENGINE TESTS
# ---------------------------------------------------------
section("2. SCORING ENGINE TESTS — NORMAL + EDGE CASES")

sample_animals = [
    {
        "name": "WaterRescueDog",
        "breed": "Labrador Retriever Mix",
        "age_upon_outcome_in_weeks": 52,
        "outcome_type": "Adoption",
    },
    {
        "name": "MountainRescueDog",
        "breed": "German Shepherd",
        "age_upon_outcome_in_weeks": 80,
        "outcome_type": "Transfer",
    },
    {
        "name": "SeniorDog",
        "breed": "Newfoundland",
        "age_upon_outcome_in_weeks": 320,
        "outcome_type": "Adoption",
    },
    {
        "name": "UnknownBreedDog",
        "breed": "Unknown Mix",
        "age_upon_outcome_in_weeks": 40,
        "outcome_type": "Other",
    },
]

print("\n-- NORMAL SCORE COMPUTATION --")
for animal in sample_animals:
    print(animal["name"], compute_rescue_score(animal))

print("\n-- RANKING SAMPLE ANIMALS --")
for idx, animal in enumerate(rank_animals(sample_animals), start=1):
    print(idx, animal["name"], animal["rescue_score"])

print("\n-- NORMALIZED BREEDS --")
for animal in sample_animals:
    print(animal["name"], normalize_breeds(animal["breed"]))

# 2.1 Scoring edge cases
section("2.1 SCORING EDGE CASES")

scoring_edge_cases = [
    {"name": "MissingBreed", "age_upon_outcome_in_weeks": 60, "outcome_type": "Adoption"},
    {"name": "MissingAge", "breed": "Labrador Retriever", "outcome_type": "Adoption"},
    {"name": "MissingOutcome", "breed": "German Shepherd", "age_upon_outcome_in_weeks": 80},
    {"name": "NegativeAge", "breed": "Newfoundland", "age_upon_outcome_in_weeks": -10, "outcome_type": "Adoption"},
    {"name": "HugeAge", "breed": "Newfoundland", "age_upon_outcome_in_weeks": 5000, "outcome_type": "Adoption"},
    {"name": "EmptyBreedString", "breed": "", "age_upon_outcome_in_weeks": 40, "outcome_type": "Other"},
    {"name": "NoneFields", "breed": None, "age_upon_outcome_in_weeks": None, "outcome_type": None},
]

print("\n-- EDGE CASE SCORE COMPUTATION --")
for animal in scoring_edge_cases:
    try:
        print(animal["name"], compute_rescue_score(animal))
    except Exception as e:
        print(animal["name"], "FAILED (expected for some cases):", e)

print("\n-- EDGE CASE BREED NORMALIZATION --")
for animal in scoring_edge_cases:
    try:
        print(animal["name"], normalize_breeds(animal.get("breed", "")))
    except Exception as e:
        print(animal["name"], "FAILED:", e)


# ---------------------------------------------------------
# 3. DATABASE AGGREGATION TESTS
# ---------------------------------------------------------
section("3. DATABASE AGGREGATION TESTS")

print("\n-- BREED DISTRIBUTION --")
try:
    for doc in shelter.aggregate_breed_distribution()[:10]:
        print(doc)
except Exception as e:
    print("Breed aggregation failed:", e)

print("\n-- OUTCOME DISTRIBUTION --")
try:
    for doc in shelter.aggregate_outcome_distribution():
        print(doc)
except Exception as e:
    print("Outcome aggregation failed:", e)

print("\n-- AGE STATISTICS --")
try:
    for doc in shelter.aggregate_age_statistics():
        print(doc)
except Exception as e:
    print("Age aggregation failed:", e)

print("\n-- RESCUE AVAILABILITY --")
try:
    for doc in shelter.aggregate_rescue_availability()[:10]:
        print(doc)
except Exception as e:
    print("Rescue availability aggregation failed:", e)

# 3.1 Filtered subset aggregation
section("3.1 FILTERED SUBSET AGGREGATION")

try:
    dogs = shelter.read({"animal_type": "Dog"})
    print("Dog subset size:", len(dogs))

    breed_counts = {}
    for doc in dogs:
        breed = doc.get("breed", "Unknown")
        breed_counts[breed] = breed_counts.get(breed, 0) + 1

    print("Top 10 dog breeds:")
    for breed, count in list(breed_counts.items())[:10]:
        print(breed, count)
except Exception as e:
    print("Filtered subset aggregation failed:", e)


# ---------------------------------------------------------
# 4. SUMMARY
# ---------------------------------------------------------
section("4. TEST SUITE COMPLETED")

print("All major CRUD, scoring, and aggregation paths exercised successfully.")
print("Review any exceptions printed above to confirm expected validation behavior.")


---- TESTING FAILED LOGIN ATTEMPT ----
Error connecting to MongoDB: Authentication failed., full error: {'ok': 0.0, 'errmsg': 'Authentication failed.', 'code': 18, 'codeName': 'AuthenticationFailed'}
Failed Login Attempt: Authentication failed., full error: {'ok': 0.0, 'errmsg': 'Authentication failed.', 'code': 18, 'codeName': 'AuthenticationFailed'}

---- INITIALIZING CRUD MODULE AND AUTHENTICATING ----
[Connection Successful] Authenticated as 'aacuser' → Database: 'aac', Collection: 'animals'

---- INSERTING STANDARD TEST RECORD ----
Create Result: True

---- READING STANDARD TEST RECORD ----
{'_id': ObjectId('6aa066a7b38d4232e2d0f40e'), 'name': 'NotebookTestAnimal', 'animal_type': 'Dog', 'breed': 'TestBreed', 'age_upon_outcome': '3 years', 'outcome_type': 'Adoption'}

---- INSERTING REALISTIC ANIMAL RECORD ----
Create Result: True

---- READING REALISTIC ANIMAL RECORD ----
{'_id': ObjectId('6aa066a7b38d4232e2d0f40f'), 'name': 'Sabrina Miranda', 'animal_type': 'Cat', 'breed': 'Domes